# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook demonstrates loading and exploring the FAIRˆ² dataset using the `mlcroissant` library, in accordance with the Croissant schema.

### Dataset Source
The dataset is defined by its Croissant schema, available at:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Define the dataset Croissant JSON-LD schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load the dataset metadata and structure
dataset = mlc.Dataset(croissant_url)

metadata = dataset.metadata

print(f"{metadata.name}:\n{metadata.description}\n")
print(f"License: {metadata.license}\nKeywords: {getattr(metadata, 'keywords', None)}\n")
print(f"Temporal coverage: {getattr(metadata, 'temporalCoverage', None)}\n" )
print(f"Spatial coverage: {getattr(metadata, 'spatialCoverage', None)}\n")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

The Croissant schema organizes data using `RecordSet` entities, which contain fields (columns). We'll enumerate all available record sets and their fields by `@id`.

In [ ]:
record_sets = list(dataset.record_sets)

if not record_sets:
    print("No record sets were found in this dataset schema.")
else:
    print(f"Number of record sets: {len(record_sets)}\n")
    for rs in record_sets:
        print(f"Record Set: {rs['@id']}  [name: {rs.get('name','(no name)')}]" )
        fields = rs.get('field', [])
        # 'field' can be a dict or list of dicts (Croissant variability)
        if isinstance(fields, dict):
            fields = [fields]
        if not fields:
            print("    No fields found for this record set.")
        else:
            for f in fields:
                print(f"    Field: {f['@id']}, name: {f.get('name',f['@id'])}, type: {f.get('dataType', None)}")

## 3. Data Extraction
Let's extract data from the record sets. All references use the Croissant entity `@id`s exactly as illustrated above. We load each record set into a pandas DataFrame and preview its data and columns.

In [ ]:
import pprint

if not record_sets:
    print("There are no data tables in this dataset. Please check the Croissant schema `recordSet` entry.")
else:
    # List of available record set @ids
    record_set_ids = [rs['@id'] for rs in record_sets]
    dataframes = {}
    for record_set_id in record_set_ids:
        # Load records for each record set
        try:
            records = list(dataset.records(record_set=record_set_id))
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Loaded '{record_set_id}' ({len(df)} records, {len(df.columns)} columns)")
            print(f"Columns: {list(df.columns)}\n")
            print(df.head(3))
        except Exception as ex:
            print(f"Could not load record set {record_set_id}: {ex}")

## 4. Exploratory Data Analysis (EDA)
We'll select a numeric field (using its field `@id`) and apply basic analyses:
- Filter records with values above a threshold,
- Normalize the field,
- Optionally group by another field.

> All references to fields and columns use their `@id`.

In [ ]:
# Pick an example record set and numeric field by @id
if not record_sets:
    print("No record sets found for EDA.")
else:
    # We pick the first available record set for demo
    record_set_id = record_set_ids[0]
    df = dataframes[record_set_id]
    
    # Find potential numeric columns by @id (infer type if missing)
    numeric_fields = []
    record_set_entry = next((rs for rs in record_sets if rs['@id'] == record_set_id), None)
    if record_set_entry:
        fields = record_set_entry.get('field', [])
        if isinstance(fields, dict):
            fields = [fields]
        for f in fields:
            if f.get('dataType', None) in ['schema:Float', 'schema:Number', 'schema:Integer', 'Float', 'Number', 'Integer']:
                numeric_fields.append(f['@id'])
    # If metadata info missing, infer from DataFrame
    if not numeric_fields:
        # Try all columns that are dtype float or int
        for col in df.columns:
            if pd.api.types.is_numeric_dtype(df[col]):
                numeric_fields.append(col)
    
    if not numeric_fields:
        print("No numeric fields detected in selected record set.")
    else:
        numeric_field_id = numeric_fields[0]
        print(f"Using numeric field: {numeric_field_id}\n")
        # Remove NaNs for analysis
        available = df[~df[numeric_field_id].isnull()]
        # Set an example threshold
        threshold = available[numeric_field_id].mean() if len(available) else 0
        # Filtered records (greater than threshold)
        filtered_df = available[available[numeric_field_id] > threshold]
        print(f"Filtered records where '{numeric_field_id}' > {threshold:.3f} (threshold is mean):\n{filtered_df.head(3)}\n")
        # Normalize
        col_norm = f"{numeric_field_id}_normalized"
        filtered_df[col_norm] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized '{numeric_field_id}':\n{filtered_df[[numeric_field_id, col_norm]].head(3)}\n")
        # Try grouping by another (non-numeric) field
        nonnum_fields = [c for c in df.columns if not pd.api.types.is_numeric_dtype(df[c])]
        group_field = nonnum_fields[0] if nonnum_fields else None
        if group_field is not None:
            grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
            print(f"Grouped by '{group_field}':\n{grouped_df.head(3)}")
        else:
            print("No suitable non-numeric group field found.")

## 5. Visualization
Visualize numeric distributions or relationships using the field `@id`s.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if not record_sets or not numeric_fields:
    print("No record sets or numeric fields available for visualization.")
else:
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=20)
    plt.title(f"Distribution of '{numeric_field_id}' in record set '{record_set_id}'")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()
    
    # If we grouped, show a bar plot of group means
    if 'grouped_df' in locals() and group_field is not None:
        plt.figure(figsize=(8,5))
        sns.barplot(data=grouped_df, x=group_field, y=numeric_field_id)
        plt.title(f"Mean of '{numeric_field_id}' grouped by '{group_field}'")
        plt.xlabel(group_field)
        plt.ylabel(f"Mean of {numeric_field_id}")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

## 6. Conclusion
In this notebook, we loaded the dataset defined by a Croissant schema, explored its metadata and structure, and dynamically referenced all entities by their `@id`. We loaded the available record sets into DataFrames, performed basic analyses and normalization on numeric fields, and visualized their distributions.

For deeper analysis, refer to the dataset [source page](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json) to review the full Croissant specification, and experiment with custom analyses using the `mlcroissant` library. All operations should always reference entities using their `@id` to ensure consistent linkage to dataset definitions.